In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt 
from sklearn.ensemble import GradientBoostingClassifier
from scipy.stats import mannwhitneyu, shapiro, kstest,f_oneway,kruskal, norm, sem
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score, confusion_matrix,recall_score,ConfusionMatrixDisplay,roc_curve, auc
from itertools import cycle
import random
from itertools import chain
import shap
from sklearn.linear_model import Lasso



np.random.seed(27)

# funzioni

In [ ]:
def scaling_robust (x_train, x_test):
    # fit scaler on training data
    scaler = RobustScaler().fit(x_train)
    # apply scaler on training and test data
    x_train_scaled = pd.DataFrame(scaler.transform(x_train))
    x_test_scaled = pd.DataFrame(scaler.transform(x_test))
    x_train_scaled.columns = x_train.columns
    x_test_scaled.columns = x_train.columns
    
    return (x_train_scaled,x_test_scaled)

In [ ]:
def resampling_smote (x_train,y_train):
    x_train_resampled, y_train_resampled = SMOTE(random_state=3,sampling_strategy='not majority' ).fit_resample(x_train, y_train)
    
    return (x_train_resampled,y_train_resampled)

In [ ]:
# mann whitney U-test - binary task
def feat_sel_mw (x_train,y_train,x_test):
    # training set per class
    y_train.reset_index(drop=True, inplace = True)
    class0 = x_train.loc[y_train == 0]
    class1 = x_train.loc[y_train == 1]
    # perform test
    U1, pv = mannwhitneyu(class0, class1)
    # select feature with pvalue<0.01
    train_selected_features_mw = x_train.iloc[:,pv<0.01]
    test_selected_features_mw = x_test.iloc[:,pv<0.01]
    # if no feature with pvalue<0.01 -> take the first 30 with smallest pvalue
    if train_selected_features_mw.shape[1] == 0:
        #pvalue descending order
        index_min_pv = np.argsort(-pv)
        # create final df
        train_selected_features_mw = pd.DataFrame()
        test_selected_features_mw = pd.DataFrame()
        for i in range(31):
            col_i = x_train.columns[index_min_pv[i]]
            train_selected_features_mw[col_i] = x_train[[col_i]]
            test_selected_features_mw[col_i] = x_test[[col_i]]

    return (train_selected_features_mw,test_selected_features_mw)

In [ ]:
# Kruskal-Wallis test or one-way ANOVA test - multiclass task
def feat_sel_stat (x_train, y_train, x_test):
    y_train.reset_index(drop=True, inplace = True)
    class0 = x_train.loc[y_train == 0]
    class1 = x_train.loc[y_train == 1]
    class2 = x_train.loc[y_train == 2]
    pvf = []
#small sample -> shapiro normality test
    if len(y_train) < 50: 
        S0, pv0 = shapiro(class0)
        S1, pv1 = shapiro(class1)
        S2, pv2 = shapiro(class2)
#normality -> one way anova test
        if min(pv0,pv1,pv2)>0.01: 
            for st in range(np.shape(class0)[1]):
                sa,pva = f_oneway(class0.iloc[:,st],class1.iloc[:,st],class2.iloc[:,st])
                pvf.append(pva)
            check = pvf[pvf<0.01]
            if len(check) == 0:
                index_min_pv = np.argsort(-pvf)
                x_train_selected_stat = x_train[[x_train.columns[index_min_pv[0:31]]]]
                x_test_selected_stat = x_test[[x_train.columns[index_min_pv[0:31]]]]
            else:
                x_train_selected_stat = x_train.iloc[:,pvf<0.01]
                x_test_selected_stat = x_test.iloc[:,pvf<0.01]
#non-normality -> kruscal wallis test
        else: 
            sk,pvf = kruskal(class0,class1,class2)
            check = pvf[pvf<0.01]
            if len(check) == 0:
                index_min_pv = np.argsort(-pvf)
                x_train_selected_stat = x_train[[x_train.columns[index_min_pv[0:31]]]]
                x_test_selected_stat = x_test[[x_train.columns[index_min_pv[0:31]]]]
            else:
                x_train_selected_stat = x_train.iloc[:,pvf<0.01]
                x_test_selected_stat = x_test.iloc[:,pvf<0.01]
#bigger sample -> kolmogorov smirnov normality test
    else:
        pv = []
        for r in range(np.shape(class0)[1]):
            res0 = kstest(class0.iloc[:,r], 'norm')
            pv.append(res0[1])
            res1 = kstest(class1.iloc[:,r],'norm')
            pv.append(res1[1])
            res2 = kstest(class2.iloc[:,r],'norm')
            pv.append(res2[1])
#normality -> one way anova test
        if min(pv)>0.01: 
            for st in range(np.shape(class0)[1]):
                sa,pva = f_oneway(class0.iloc[:,st],class1.iloc[:,st],class2.iloc[:,st])
                pvf.append(pva)
            check = pvf[pvf<0.01]
            if len(check) == 0:
                index_min_pv = np.argsort(-pvf)
                x_train_selected_stat = x_train[[x_train.columns[index_min_pv[0:31]]]]
                x_test_selected_stat = x_test[[x_train.columns[index_min_pv[0:31]]]]
            else:
                x_train_selected_stat = x_train.iloc[:,pvf<0.01]
                x_test_selected_stat = x_test.iloc[:,pvf<0.01]
#non-normality -> kruscal wallis test
        else:
            sk,pvf = kruskal(class0,class1,class2)
            check = pvf[pvf<0.01]
            if len(check) == 0:
                index_min_pv = np.argsort(-pvf)
                x_train_selected_stat = x_train[[x_train.columns[index_min_pv[0:31]]]]
                x_test_selected_stat = x_test[[x_train.columns[index_min_pv[0:31]]]]
            else:
                x_train_selected_stat = x_train.iloc[:,pvf<0.01]
                x_test_selected_stat = x_test.iloc[:,pvf<0.01]

    return (x_train_selected_stat,x_test_selected_stat)

In [ ]:
def feat_sel_lasso (x_train,y_train,x_test, alpha): 
    # Fit Lasso model
    lasso = Lasso(alpha=alpha)
    lasso.fit(x_train, y_train)
    # Select features with non-zero coefficients
    selected_features = x_train.columns[(lasso.coef_ != 0)]
    # Create new DataFrames with selected features
    x_train_selected = x_train[selected_features]
    x_test_selected = x_test[selected_features]

    return (x_train_selected,x_test_selected)

In [ ]:
# feature selection binary task
def feat_sel(x_train, x_test, y_train, alpha):
    x_train_mw, x_test_mw = feat_sel_mw (x_train,y_train,x_test)
    n_feat_stat = np.shape(x_train_mw)[1]

    x_train_mw_lasso,x_test_mw_lasso = feat_sel_lasso(x_train_mw,y_train,x_test_mw,alpha)
    n_feat = np.shape(x_train_mw_lasso)[1]

    print(f'binary n feat stat: {n_feat_stat} - n feat sel lasso: {n_feat} - alpha:{alpha}')    
    return (x_train_mw_lasso,x_test_mw_lasso)


In [ ]:
# feature selection multiclass task
def feat_sel_multi(x_train, x_test, y_train, alpha):
    x_train_stat, x_test_stat = feat_sel_stat (x_train, y_train, x_test)
    n_feat_stat = np.shape(x_train_stat)[1]

    x_train_stat_lasso, x_test_stat_lasso = feat_sel_lasso(x_train_stat, y_train, x_test_stat, alpha)
    n_feat = np.shape(x_train_stat_lasso)[1]

    print(f'0-1234-5 n feat stat: {n_feat_stat} - n feat sel lasso: {n_feat} - alpha:{alpha}') 
    return (x_train_stat_lasso, x_test_stat_lasso)

In [ ]:
def training_set_creation(pat_df,training_idx,patients,X,y):
    i = 0
    tr = pat_df.iloc[training_idx] # id patients training crossvalidation
    for pt in range(len(tr)):
        pat_tr = ((str(tr.iloc[pt])).split())[1]#id patient
        indices_tr = [i for i, s in enumerate(patients) if pat_tr in s]# trainig indices in X and y corresponding to selected patient 
        x_rows_pat_tr = X.iloc[indices_tr]# X rows in training
        y_rows_pat_tr = y.iloc[indices_tr]# y rows in training
        if i == 0:#first patient -> initialize
            x_train_fold = x_rows_pat_tr
            y_train_fold = y_rows_pat_tr
            i = 1
        else:# from the second patient on -> concatente  
            x_train_fold = pd.concat([x_train_fold,x_rows_pat_tr]) 
            y_train_fold = pd.concat([y_train_fold,y_rows_pat_tr])
    return (x_train_fold,y_train_fold)



In [ ]:
def validation_set_creation (pat_df,y_train_pat,validation_idx,patients,X,y):
    i = 0
    te = pat_df.iloc[validation_idx] # id patients training crossvalidation
    y_validation_pat = y_train_pat[validation_idx] 
    imgXpat = []
    for pt in range(len(te)):
        pat_te = ((str(te.iloc[pt])).split())[1]
        indices_te = [i for i, s in enumerate(patients) if pat_te in s]# validation indices in X corresponding to selected patient 
        x_rows_pat_te = X.iloc[indices_te]# X rows in validation
        if i == 0: #first patient -> initialize
            x_validation_fold = x_rows_pat_te
            imgXpat.append(len(indices_te))
            i = 1
        else:# from the second patient on -> concatente
            x_validation_fold = pd.concat([x_validation_fold,x_rows_pat_te]) 
            imgXpat.append(len(indices_te))

    return(x_validation_fold,y_validation_pat,imgXpat)


In [ ]:
# def major_voting(y_pred,y_prob,imgXpat, n_classes): 
#     y_pred_cor_mv = []
#     y_prob_pat_mean_sum = []
#     y_prob_pat_mean_0 = []
#     y_prob_pat_mean_1 = []
#     y_prob_pat_mean_2 = []

#     n_img = 0
#     for n in imgXpat: #number of images per patient
#         if n_img < sum(imgXpat):# avoid error
#             # predictions
#             y_pred_pat = y_pred[n_img : n_img + n] # predictions corresponding to the patient considered
#             numerosity_classes=[]
#             for i in range(n_classes-1, -1,-1):
#                 numerosity_classes.append((y_pred_pat == i).sum()) # count number of predictions per class, descendent order of classes such that in case of tie the worst class is selected
#             class_max = (n_classes-1) - (numerosity_classes.index(max(numerosity_classes)))
#             y_pred_cor_mv.append(class_max)
#             # prediction probabilities
#             y_prob_pat = y_prob[n_img:n_img+n,:]    # prediction probabilities corresponding to the patient considered                   
#             y_prob_pat_mean = y_prob_pat.mean(axis=0)# perform the mean per patient (and per class)
#             tot = y_prob_pat_mean.sum()# calculate the total sum of probabilities resulted from mean (the sum of prediction probabilities of the classes must be equal to 1)
#             if n_classes>2: #multiclass
#                 y_prob_pat_mean_sum.append([x / tot for x in y_prob_pat_mean]) 
#                 y_prob_pat_mean_0.append(y_prob_pat_mean[0]/tot)
#                 y_prob_pat_mean_1.append(y_prob_pat_mean[1]/tot) 
#                 y_prob_pat_mean_2.append(y_prob_pat_mean[2]/tot)
#             else: #binary
#                 y_prob_pat_mean_sum.append([x / tot for x in y_prob_pat_mean])
#                 y_prob_pat_mean_0.append(y_prob_pat_mean[0]/tot)
#                 y_prob_pat_mean_1.append(y_prob_pat_mean[1]/tot) 
                                           
#             n_img = n_img + n

#     return (y_pred_cor_mv,y_prob_pat_mean_sum,y_prob_pat_mean_0,y_prob_pat_mean_1,y_prob_pat_mean_2)


## Majority Voting Multiclass

In [ ]:
# Ronchetti

# import numpy as np

# def major_voting_multiclass(y_pred, y_prob, imgXpat): 
#     """
#     Versione finale pulita che restituisce solo:
#     - Predizione finale (major voting)  
#     - Probabilità paziente (prima / (prima + seconda))
#     """
#     y_pred_cor_mv = []
#     y_prob_final_ratio = []

#     n_img = 0
#     n_classes = 3  
#     for n in imgXpat:
#         if n_img < sum(imgXpat):
            
#             # STEP 1: MAJOR VOTING
#             y_pred_pat = y_pred[n_img : n_img + n]
#             numerosity_classes = []
#             for i in range(n_classes-1, -1, -1):
#                 numerosity_classes.append((y_pred_pat == i).sum())
#             class_max = (n_classes-1) - (numerosity_classes.index(max(numerosity_classes)))
#             y_pred_cor_mv.append(class_max)
            
#             # STEP 2: CALCOLA MEDIA PROBABILITÀ PER CLASSE PREDETTA
#             y_prob_pat = y_prob[n_img:n_img+n, :]
#             mean_prob_by_predicted_class = {}
            
#             for classe in range(n_classes):
#                 mask_predicted_as_class = (y_pred_pat == classe)
#                 if np.sum(mask_predicted_as_class) > 0:
#                     prob_images_predicted_as_class = y_prob_pat[mask_predicted_as_class, :]
#                     mean_prob_for_class = np.mean(prob_images_predicted_as_class, axis=0)
#                     mean_prob_by_predicted_class[classe] = mean_prob_for_class
#                 else:
#                     mean_prob_by_predicted_class[classe] = np.zeros(n_classes)
            
#             # STEP 3: PRENDI MEDIA DELLA CLASSE VINCENTE E CALCOLA LA OREDIZIONE FINALE COME RATIO TRA
#             # LA SUA PROBABILITA E QUELLA RELATIVA ALLA CLASSE COMPETITOR
#             winning_class_mean_probs = mean_prob_by_predicted_class[class_max]
#             sorted_probs = np.sort(winning_class_mean_probs)[::-1]
            
#             if len(sorted_probs) >= 2:
#                 ratio = sorted_probs[0] / (sorted_probs[0] + sorted_probs[1])
#             else:
#                 ratio = 1.0
                
#             y_prob_final_ratio.append(ratio)
#             n_img = n_img + n

#     return y_pred_cor_mv, y_prob_final_ratio



In [ ]:
import numpy as np

# Dividi per la media
def major_voting_multiclass(y_pred, y_prob, imgXpat): 
    """
    Versione finale pulita che restituisce solo:
    - Predizione finale (major voting)  
    - Probabilità paziente (prima / (prima + seconda))
    """
    y_pred_cor_mv = []
    y_prob_final_ratio = []

    n_img = 0
    n_classes = 3  
    for n in imgXpat:
        if n_img < sum(imgXpat):
            
            # STEP 1: MAJOR VOTING
            y_pred_pat = y_pred[n_img : n_img + n]
            numerosity_classes = []
            for i in range(n_classes-1, -1, -1):
                numerosity_classes.append((y_pred_pat == i).sum())
            class_max = (n_classes-1) - (numerosity_classes.index(max(numerosity_classes)))
            y_pred_cor_mv.append(class_max)
            
            # STEP 2: CALCOLA MEDIA PROBABILITÀ PER CLASSE PREDETTA
            y_prob_pat = y_prob[n_img:n_img+n, :]
            mean_prob_by_predicted_class = {}
            
            for classe in range(n_classes):
                mask_predicted_as_class = (y_pred_pat == classe)
                if np.sum(mask_predicted_as_class) > 0:
                    prob_images_predicted_as_class = y_prob_pat[mask_predicted_as_class, :]
                    mean_prob_for_class = np.mean(prob_images_predicted_as_class, axis=0)
                    mean_prob_by_predicted_class[classe] = mean_prob_for_class
                else:
                    mean_prob_by_predicted_class[classe] = np.zeros(n_classes)
            
            # STEP 3: PRENDI MEDIA DELLA CLASSE VINCENTE E CALCOLA LA OREDIZIONE FINALE COME RATIO TRA
            # LA SUA PROBABILITA E QUELLA RELATIVA ALLA MEDIA DELLE ALTRE DUE
            winning_class_mean_probs = mean_prob_by_predicted_class[class_max]
            sorted_probs = np.sort(winning_class_mean_probs)[::-1]


            # Prima probabilità (più alta) e seconda probabilità
            first_prob = sorted_probs[0]
            second_prob = sorted_probs[1]
            
            # Ratio = prima / (prima + seconda)
            ratio = first_prob / (first_prob + second_prob)
            
                
            y_prob_final_ratio.append(ratio)
            n_img = n_img + n

    return y_pred_cor_mv, y_prob_final_ratio



## Majority Voting Multiclass - Validation

In [ ]:
import numpy as np

# Dividi per la media
def major_voting_multiclass_validation(y_pred, y_prob, imgXpat): 
    """
    Versione finale pulita che restituisce solo:
    - Predizione finale (major voting)  
    - Probabilità paziente (prima / (prima + seconda))
    """
    y_pred_cor_mv = []
    y_prob_final_ratio = []

    n_img = 0
    n_classes = 3  
    for n in imgXpat:
        if n_img < sum(imgXpat):
            
            # STEP 1: MAJOR VOTING
            y_pred_pat = y_pred[n_img : n_img + n]
            numerosity_classes = []
            for i in range(n_classes-1, -1, -1):
                numerosity_classes.append((y_pred_pat == i).sum())
            class_max = (n_classes-1) - (numerosity_classes.index(max(numerosity_classes)))
            y_pred_cor_mv.append(class_max)
            
            # STEP 2: CALCOLA MEDIA PROBABILITÀ PER CLASSE PREDETTA
            y_prob_pat = y_prob[n_img:n_img+n, :]
            mean_prob_by_predicted_class = {}
            
            for classe in range(n_classes):
                mask_predicted_as_class = (y_pred_pat == classe)
                if np.sum(mask_predicted_as_class) > 0:
                    prob_images_predicted_as_class = y_prob_pat[mask_predicted_as_class, :]
                    mean_prob_for_class = np.mean(prob_images_predicted_as_class, axis=0)
                    mean_prob_by_predicted_class[classe] = mean_prob_for_class
                else:
                    mean_prob_by_predicted_class[classe] = np.zeros(n_classes)
            
            # STEP 3: PRENDI MEDIA DELLA CLASSE VINCENTE E CALCOLA LA OREDIZIONE FINALE COME
            # 1-MEDIA(PROBABILITA NON VINCENTI)
            winning_class_mean_probs = mean_prob_by_predicted_class[class_max]

                
            y_prob_final_ratio.append(winning_class_mean_probs)
            n_img = n_img + n

    return y_pred_cor_mv, y_prob_final_ratio



## Majority Voting Binary

In [ ]:
import numpy as np

def major_voting_binary(y_pred, y_prob, imgXpat): 
    """
    Major voting per task BINARIO (massimo 2 classi)
    
    Returns:
    - Predizione finale (major voting)  
    - Probabilità paziente (prima / (prima + seconda)) - per task binario è semplicemente la prob della classe vincente
    """
    y_pred_cor_mv = []
    y_prob_final_ratio = []
    
    n_classes = 2  # Fisso per task binario

    n_img = 0
    for n in imgXpat:
        if n_img < sum(imgXpat):
            
            # STEP 1: MAJOR VOTING
            y_pred_pat = y_pred[n_img : n_img + n]
            numerosity_classes = []
            for i in range(n_classes-1, -1, -1):  # Da classe 1 a 0
                numerosity_classes.append((y_pred_pat == i).sum())
            class_max = (n_classes-1) - (numerosity_classes.index(max(numerosity_classes)))
            y_pred_cor_mv.append(class_max)
            
            # STEP 2: CALCOLA MEDIA PROBABILITÀ PER CLASSE PREDETTA
            y_prob_pat = y_prob[n_img:n_img+n, :]
            mean_prob_by_predicted_class = {}
            
            for classe in range(n_classes):
                mask_predicted_as_class = (y_pred_pat == classe)
                if np.sum(mask_predicted_as_class) > 0:
                    prob_images_predicted_as_class = y_prob_pat[mask_predicted_as_class, :]
                    mean_prob_for_class = np.mean(prob_images_predicted_as_class, axis=0)
                    mean_prob_by_predicted_class[classe] = mean_prob_for_class
                else:
                    mean_prob_by_predicted_class[classe] = np.zeros(n_classes)
            
            # STEP 3: PRENDI PROBABILITÀ DELLA CLASSE VINCENTE
            # Per task binario, la probabilità finale è semplicemente quella della classe vincente
            winning_class_mean_probs = mean_prob_by_predicted_class[class_max]
            
            # La probabilità della classe vincente
            prob_classe_vincente = winning_class_mean_probs[class_max]
            
            y_prob_final_ratio.append(prob_classe_vincente)
            n_img = n_img + n

    return y_pred_cor_mv, y_prob_final_ratio

## Majority Voting Binary - Validation

In [ ]:
import numpy as np

def major_voting_binary_validation(y_pred, y_prob, imgXpat): 
    """
    Major voting per task BINARIO (massimo 2 classi)
    
    Returns:
    - Predizione finale (major voting)  
    - Vettore probabilità per ogni paziente [prob_classe_0, prob_classe_1]
    """
    y_pred_cor_mv = []
    y_prob_final_ratio = []
    
    n_classes = 2  # Fisso per task binario

    n_img = 0
    for n in imgXpat:
        if n_img < sum(imgXpat):
            
            # STEP 1: MAJOR VOTING
            y_pred_pat = y_pred[n_img : n_img + n]
            numerosity_classes = []
            for i in range(n_classes-1, -1, -1):  # Da classe 1 a 0
                numerosity_classes.append((y_pred_pat == i).sum())
            class_max = (n_classes-1) - (numerosity_classes.index(max(numerosity_classes)))
            y_pred_cor_mv.append(class_max)
            
            # STEP 2: CALCOLA MEDIA PROBABILITÀ PER CLASSE PREDETTA
            y_prob_pat = y_prob[n_img:n_img+n, :]
            mean_prob_by_predicted_class = {}
            
            for classe in range(n_classes):
                mask_predicted_as_class = (y_pred_pat == classe)
                if np.sum(mask_predicted_as_class) > 0:
                    prob_images_predicted_as_class = y_prob_pat[mask_predicted_as_class, :]
                    mean_prob_for_class = np.mean(prob_images_predicted_as_class, axis=0)
                    mean_prob_by_predicted_class[classe] = mean_prob_for_class
                else:
                    mean_prob_by_predicted_class[classe] = np.zeros(n_classes)
            
            # STEP 3: RESTITUISCI VETTORE COMPLETO DELLE PROBABILITÀ
            # Restituisce il vettore completo [prob_classe_0, prob_classe_1]
            winning_class_mean_probs = mean_prob_by_predicted_class[class_max]
            
            # Restituisci l'intero vettore invece del singolo valore
            y_prob_final_ratio.append(winning_class_mean_probs.tolist())  # Converte in lista se è numpy array
            
            n_img = n_img + n

    return y_pred_cor_mv, y_prob_final_ratio

In [ ]:
def calculate_specificity(true_labels, predicted_labels, n_classes):
    if n_classes > 2:# multiclass task
        specificity_scores = []
        totTP = 0
        totTN = 0
        totFP = 0
        totFN = 0
        for i in range(n_classes):
            cm = confusion_matrix(true_labels, predicted_labels, labels=np.arange(0, n_classes, step=1))
            tp = cm[i,i]
            fn = sum(cm[i,:])-tp
            fp = sum([c[i] for c in cm]) - tp
            tn = (sum(sum(cm))-(tp+fn+fp))
            specificity_scores.append(tn / (tn + fp) if (tn + fp) != 0 else 0)

            totTP = totTP + cm[i,i]
            totFN = totFN + sum(cm[i,:])-tp
            totFP = totFP + sum([c[i] for c in cm]) - tp
            totTN = totTN + (sum(sum(cm))-(tp+fn+fp))

        specificity = sum(specificity_scores) / n_classes
        specificity_micro = totTN/(totTN+totFP)
    else: # binary task
        tn, fp, *_ = confusion_matrix(true_labels, predicted_labels, labels=[0,1]).ravel()
        specificity_micro = np.nan
        specificity = tn / (tn + fp)
    
    
    return specificity, specificity_micro

# main

In [ ]:
#Load data
df = pd.read_csv('features_CADRADS_original_wavelet_exponential.csv', index_col=False)

# lit of all patients (id images)
patients = []
coronaries = []
for i in range(np.shape(df)[0]):
    patients.append(f'{df["ID"][i][2]}-{df["ID"][i][8:10]}')
    coronaries.append(f'{df["ID"][i][2]}-{df["ID"][i][8:10]}-{df["ID"][i][3:6]}') 
df.insert(0,'ID_new', patients)
df.insert(1,'ID_new_cor',coronaries)
pat_list = df.sort_values(by='ID_new_cor')

# label unique patients
y_pat_list = [p[0] for p in pat_list]
y_pat_list = pd.DataFrame(y_pat_list)
y_pat_list.columns = ["CR"]

## train-test split 

In [ ]:
train_pat, test_pat, y_train_pat, y_test_pat = train_test_split(pat_list, y_pat_list, train_size= 0.9, random_state=49, stratify= y_pat_list)
train_pat_df = pd.DataFrame(train_pat)
y_train_pat = np.array(y_train_pat, dtype=int) 
y_test_pat = np.array(y_test_pat, dtype=int) 


In [ ]:
# create training sets for each task

# 0-1234-5
y_train_pat_0_1234_5 = y_train_pat.copy()
y_train_pat_0_1234_5[(y_train_pat == 2)|(y_train_pat == 3)|(y_train_pat == 4)] = 1
y_train_pat_0_1234_5[y_train_pat == 5] = 2

## 12-34
filterr = [(y_train_pat == 1) | (y_train_pat == 2) | (y_train_pat == 3)| (y_train_pat == 4)][0]
train_pat_12_34 = [i for indx,i in enumerate(train_pat) if filterr[indx] == True]
train_pat_12_34_df = pd.DataFrame(train_pat_12_34)
y_train_pat_12_34 = [i for indx,i in enumerate(y_train_pat) if filterr[indx] == True]
y_train_pat_12_34 = np.concatenate(y_train_pat_12_34)
y_train_pat_12_34[(y_train_pat_12_34 == 1) | (y_train_pat_12_34 == 2)] = 0
y_train_pat_12_34[(y_train_pat_12_34 == 3) | (y_train_pat_12_34 == 4)] = 1

### 1-2
filter12 = [(y_train_pat == 1) | (y_train_pat == 2)][0]
train_pat_12 = [i for indx,i in enumerate(train_pat) if filter12[indx] == True] 
train_pat_12_df = pd.DataFrame(train_pat_12)
y_train_pat_12 = [i for indx,i in enumerate(y_train_pat) if filter12[indx] == True]
y_train_pat_12 = np.concatenate(y_train_pat_12)
y_train_pat_12[(y_train_pat_12 == 1)] = 0
y_train_pat_12[(y_train_pat_12 == 2)] = 1

### 3-4
filter34 = [(y_train_pat == 3) | (y_train_pat == 4)][0]
train_pat_34 = [i for indx,i in enumerate(train_pat) if filter34[indx] == True] 
train_pat_34_df = pd.DataFrame(train_pat_34)
y_train_pat_34 = [i for indx,i in enumerate(y_train_pat) if filter34[indx] == True]
y_train_pat_34 = np.concatenate(y_train_pat_34)
y_train_pat_34[(y_train_pat_34 == 3)] = 0
y_train_pat_34[(y_train_pat_34 == 4)] = 1


In [ ]:
# define classification parameters
classifier = GradientBoostingClassifier(n_estimators=20, learning_rate=0.5, max_depth=2, random_state=0) 
fold = 5
skf = StratifiedKFold(n_splits=fold)


In [ ]:
# best lasso regulation parameter for each task (selected on crossvalidation)
k_0_1234_5 = 0.05
k_12_34 = 0.05
k_12 = 0.5
k_34 = 0.05

## Cross-Validation

### Task 0 - 1234 - 5

In [ ]:
#0-1234-5
insight_0_1234_5 = []
CM_mean_0_1234_5 = []
CM_sum_0_1234_5 = []

CM_0_1234_5 = []
cm_sum_0_1234_5 = [[0 for j in range(3)] for i in range(3)]
AUC_ROC_macro_0_1234_5 = []
AUC_ROC_micro_0_1234_5 = []
bal_acc_0_1234_5 = []
f1_macro_0_1234_5 = []
f1_micro_0_1234_5 = []
sens_macro_0_1234_5 = []
sens_micro_0_1234_5 = []
spec_macro_0_1234_5=[]
spec_micro_0_1234_5=[]

for train_0_1234_5_idx, validation_0_1234_5_idx in skf.split(train_pat_df,y_train_pat_0_1234_5):
    
    #training set crossvalidation
    x_0_1234_5_train_fold,y_0_1234_5_train_fold = training_set_creation(train_pat_df,train_0_1234_5_idx,patients_new,X,y)

    #modify label
    y_0_1234_5_train_fold.loc[(y_0_1234_5_train_fold == 2)|(y_0_1234_5_train_fold == 3)|(y_0_1234_5_train_fold == 4)] = 1
    y_0_1234_5_train_fold.loc[y_0_1234_5_train_fold == 5] = 2

    #validation set crossvalidation
    x_validation_pat_0_1234_5,y_validation_pat_fold_0_1234_5,imgXpat_0_1234_5 = validation_set_creation (train_pat_df,y_train_pat_0_1234_5,validation_0_1234_5_idx,patients_new,X,y)    

    #robust scaler
    x_train_fold_scaled, x_validation_fold_scaled = scaling_robust (x_0_1234_5_train_fold, x_validation_pat_0_1234_5)                    

    #feature selection
    x_train_fold_sel,x_validation_fold_sel = feat_sel_multi (x_train_fold_scaled, x_validation_fold_scaled, y_0_1234_5_train_fold, k_0_1234_5) 

    #smote
    x_train_fold_resample, y_train_fold_resample = resampling_smote (x_train_fold_sel, y_0_1234_5_train_fold)

    # classifier fit
    classifier.fit(x_train_fold_resample, y_train_fold_resample)

    # prediction
    y_pred = classifier.predict(x_validation_fold_sel)
    y_prob = classifier.predict_proba(x_validation_fold_sel)

    #major voting over patient
    y_pred_pat_mv_0_1234_5, y_prob_pat_mean_sum_0_1234_5 = major_voting_multiclass_validation(y_pred, y_prob, imgXpat_0_1234_5)
    #y_pred_pat_mv_0_1234_5, y_prob_pat_mean_sum_0_1234_5,*_ = major_voting(y_pred, y_prob, imgXpat_0_1234_5, 3)
    
    #metrics
    cm_sum_0_1234_5 = np.array(cm_sum_0_1234_5) + np.array(confusion_matrix(y_validation_pat_fold_0_1234_5, y_pred_pat_mv_0_1234_5))
    CM_0_1234_5.append(confusion_matrix(y_validation_pat_fold_0_1234_5, y_pred_pat_mv_0_1234_5))
    AUC_ROC_macro_0_1234_5.append(roc_auc_score(y_validation_pat_fold_0_1234_5, y_prob_pat_mean_sum_0_1234_5,average = 'macro',multi_class = 'ovr'))
    AUC_ROC_micro_0_1234_5.append(roc_auc_score(y_validation_pat_fold_0_1234_5, y_prob_pat_mean_sum_0_1234_5,average = 'micro',multi_class = 'ovr'))
    bal_acc_0_1234_5.append(balanced_accuracy_score(y_validation_pat_fold_0_1234_5, y_pred_pat_mv_0_1234_5))
    f1_macro_0_1234_5.append(f1_score(y_validation_pat_fold_0_1234_5, y_pred_pat_mv_0_1234_5, pos_label= 1,average = 'macro'))
    f1_micro_0_1234_5.append(f1_score(y_validation_pat_fold_0_1234_5, y_pred_pat_mv_0_1234_5, pos_label= 1,average = 'micro'))
    sens_macro_0_1234_5.append(recall_score(y_validation_pat_fold_0_1234_5,y_pred_pat_mv_0_1234_5, average='macro'))
    sens_micro_0_1234_5.append(recall_score(y_validation_pat_fold_0_1234_5,y_pred_pat_mv_0_1234_5, average='micro'))
    spec_macro_0_1234_5.append(calculate_specificity(y_validation_pat_fold_0_1234_5, y_pred_pat_mv_0_1234_5, n_classes = 3)[0])
    spec_micro_0_1234_5.append(calculate_specificity(y_validation_pat_fold_0_1234_5, y_pred_pat_mv_0_1234_5, n_classes = 3)[1])

    
cm_mean = np.stack(CM_0_1234_5).mean(axis=0)
CM_mean_0_1234_5.append(cm_mean)
CM_sum_0_1234_5.append(cm_sum_0_1234_5)
insight_0_1234_5.append([k_0_1234_5,
                         format(np.mean(AUC_ROC_macro_0_1234_5), ".2f"),format(np.std(AUC_ROC_macro_0_1234_5), ".2f"), 
                          format(np.mean(AUC_ROC_micro_0_1234_5), ".2f"),format(np.std(AUC_ROC_micro_0_1234_5), ".2f"), 
                          format(np.mean(bal_acc_0_1234_5), ".2f"),format(np.std(bal_acc_0_1234_5), ".2f"),
                            format(np.mean(f1_macro_0_1234_5), ".2f"),format(np.std(f1_macro_0_1234_5), ".2f"),
                            format(np.mean(f1_micro_0_1234_5), ".2f"),format(np.std(f1_micro_0_1234_5), ".2f"),
                            format(np.mean(sens_macro_0_1234_5), ".2f"),format(np.std(sens_macro_0_1234_5), ".2f"),
                            format(np.mean(sens_micro_0_1234_5), ".2f"),format(np.std(sens_micro_0_1234_5), ".2f"),
                            format(np.mean(spec_macro_0_1234_5), ".2f"),format(np.std(spec_macro_0_1234_5), ".2f"),
                            format(np.mean(spec_micro_0_1234_5), ".2f"),format(np.std(spec_micro_0_1234_5), ".2f")
                            ])
                     
insight_0_1234_5_df = pd.DataFrame(insight_0_1234_5)
insight_0_1234_5_df.columns = ["k","AUC_ROC_macro_mean", "AUC_ROC_macro_std","AUC_ROC_micro_mean", "AUC_ROC_micro_std",
                               "bal_acc_mean", "bal_acc_std",
                               "f1_macro_mean","f1_macro_std","f1_micro_mean","f1_micro_std",
                               "sensitivity_macro_mean","sensitivity_macro_std","sensitivity_micro_mean","sensitivity_micro_std",
                               "specificity_macro_mean","specificity_macro_std","specificity_micro_mean","specificity_micro_std"]


### Task 12-34

In [ ]:
#12-34
insight_12_34 = []
CM_mean_12_34  = []
CM_sum_12_34  = []

CM_12_34  = []
cm_sum_12_34  = [[0 for j in range(2)] for i in range(2)]
AUC_ROC_12_34 = []
bal_acc_12_34 = []
f1_12_34 = []
sens_12_34=[]
spec_12_34=[]

for train_12_34_idx, validation_12_34_idx in skf.split(train_pat_12_34_df,y_train_pat_12_34):

    #training set crossvalidation
    x_12_34_train_fold,y_train_fold = training_set_creation(train_pat_12_34_df,train_12_34_idx,patients_new,X,y)

    #modify label
    y_12_34_train_fold = y_train_fold.copy()
    y_12_34_train_fold.loc[(y_train_fold == 1) | (y_train_fold == 2)] = 0
    y_12_34_train_fold.loc[(y_train_fold == 3) | (y_train_fold == 4)] = 1

    #validation set crossvalidation
    x_validation_pat_12_34,y_validation_pat_12_34,imgXpat_12_34 = validation_set_creation (train_pat_12_34_df,y_train_pat_12_34,validation_12_34_idx,patients_new,X,y)        

    #robust scaler
    x_train_fold_scaled, x_validation_fold_scaled = scaling_robust (x_12_34_train_fold, x_validation_pat_12_34)         

    #feature selection
    x_train_fold_sel,x_validation_fold_sel = feat_sel(x_train_fold_scaled, x_validation_fold_scaled, y_12_34_train_fold, k_12_34) 

    #smote
    x_train_fold_resample, y_train_fold_resample = resampling_smote (x_train_fold_sel, y_12_34_train_fold)

    #classifier fitting
    classifier.fit(x_train_fold_resample, y_train_fold_resample)

    #prediction
    y_pred = classifier.predict(x_validation_fold_sel)
    y_prob = classifier.predict_proba(x_validation_fold_sel)

    #mjv patient
    y_pred_pat_mv_12_34, y_prob_pat_class1_12_34 = major_voting_binary_validation(y_pred, y_prob, imgXpat_12_34)
    #y_pred_pat_mv_12_34, y_prob_pat_mean_sum_12_34,y_prob_pat_class0_12_34,y_prob_pat_class1_12_34,*_ = major_voting(y_pred, y_prob, imgXpat_12_34, 2)

    #metrics
    cm_sum_12_34 = np.array(cm_sum_12_34) + np.array(confusion_matrix(y_validation_pat_12_34, y_pred_pat_mv_12_34))
    CM_12_34.append(confusion_matrix(y_validation_pat_12_34, y_pred_pat_mv_12_34))
    AUC_ROC_12_34.append(roc_auc_score(y_validation_pat_12_34, np.array(y_prob_pat_class1_12_34)[:, 1]))
    bal_acc_12_34.append(balanced_accuracy_score(y_validation_pat_12_34, y_pred_pat_mv_12_34))
    f1_12_34.append((f1_score(y_validation_pat_12_34, y_pred_pat_mv_12_34, pos_label= 1)))
    sens_12_34.append(recall_score(y_validation_pat_12_34,y_pred_pat_mv_12_34))
    spec_12_34.append(calculate_specificity(y_validation_pat_12_34, y_pred_pat_mv_12_34, n_classes = 2)[0])

    
CM_mean_12_34 = np.stack(CM_12_34).mean(axis=0)
CM_sum_12_34.append(cm_sum_12_34)
insight_12_34.append([k_12_34,format(np.mean(AUC_ROC_12_34), ".2f"),format(np.std(AUC_ROC_12_34), ".2f"), 
            format(np.mean(bal_acc_12_34), ".2f"),format(np.std(bal_acc_12_34), ".2f"),
            format(np.mean(f1_12_34), ".2f"),format(np.std(f1_12_34), ".2f"),
            format(np.mean(sens_12_34), ".2f"),format(np.std(sens_12_34), ".2f"),
            format(np.mean(spec_12_34), ".2f"),format(np.std(spec_12_34), ".2f")])        

insight_12_34_df = pd.DataFrame(insight_12_34)
insight_12_34_df.columns = ["k","AUC_ROC_mean", "AUC_ROC_std", "bal_acc_mean", "bal_acc_std","f1_mean","f1_std","sensitivity_mean","sensitivity_std","specificity_mean","specificity_std"]


### Task 1-2

In [ ]:
#1-2
insight_12 = []
CM_mean_12  = []
CM_sum_12  = []

CM_12  = []
cm_sum_12  = [[0 for j in range(2)] for i in range(2)]
AUC_ROC_12 = []
bal_acc_12 = []
f1_12 = []
sens_12=[]
spec_12=[]

for train_12_idx, validation_12_idx in skf.split(train_pat_12_df,y_train_pat_12):

    #training set crossvalidation
    x_12_train_fold,y_train_fold = training_set_creation(train_pat_12_df,train_12_idx,patients_new,X,y)

    #modify label
    y_12_train_fold = y_train_fold.copy()
    y_12_train_fold.loc[y_train_fold == 1] = 0
    y_12_train_fold.loc[y_train_fold == 2] = 1

    #validation set crossvalidation
    x_validation_pat_12,y_validation_pat_12,imgXpat_12 = validation_set_creation (train_pat_12_df,y_train_pat_12,validation_12_idx,patients_new,X,y)      

    #robust scaler
    x_train_fold_scaled, x_validation_fold_scaled = scaling_robust (x_12_train_fold, x_validation_pat_12)     

    #feature selection
    x_train_fold_sel,x_validation_fold_sel = feat_sel(x_train_fold_scaled, x_validation_fold_scaled, y_12_train_fold, k_12) 

    if len(x_train_fold_sel.columns) !=0:

        #smote
        x_train_fold_resample, y_train_fold_resample = resampling_smote (x_train_fold_sel, y_12_train_fold)

        #classifier fitting
        classifier.fit(x_train_fold_resample, y_train_fold_resample)

        #prediction
        y_pred = classifier.predict(x_validation_fold_sel)
        y_prob = classifier.predict_proba(x_validation_fold_sel)

        #mjv patient
        y_pred_pat_mv_12, y_prob_pat_class1_12 = major_voting_binary_validation(y_pred, y_prob, imgXpat_12)
        # y_pred_pat_mv_12, y_prob_pat_mean_sum_12,y_prob_pat_class0_12,y_prob_pat_class1_12,*_ = major_voting(y_pred, y_prob, imgXpat_12, 2)

        #metrics
        cm_sum_12 = np.array(cm_sum_12) + np.array(confusion_matrix(y_validation_pat_12, y_pred_pat_mv_12))
        CM_12.append(confusion_matrix(y_validation_pat_12, y_pred_pat_mv_12))
        AUC_ROC_12.append(roc_auc_score(y_validation_pat_12, np.array(y_prob_pat_class1_12)[:, 1]))
        bal_acc_12.append(balanced_accuracy_score(y_validation_pat_12, y_pred_pat_mv_12))
        f1_12.append((f1_score(y_validation_pat_12, y_pred_pat_mv_12, pos_label= 1)))
        sens_12.append(recall_score(y_validation_pat_12,y_pred_pat_mv_12))
        spec_12.append(calculate_specificity(y_validation_pat_12, y_pred_pat_mv_12, n_classes = 2)[0])
        
        
CM_mean_12 = np.stack(CM_12).mean(axis=0)
CM_sum_12.append(cm_sum_12)
insight_12.append([k_12,format(np.mean(AUC_ROC_12), ".2f"),format(np.std(AUC_ROC_12), ".2f"), 
            format(np.mean(bal_acc_12), ".2f"),format(np.std(bal_acc_12), ".2f"),
            format(np.mean(f1_12), ".2f"),format(np.std(f1_12), ".2f"),
            format(np.mean(sens_12), ".2f"),format(np.std(sens_12), ".2f"),
            format(np.mean(spec_12), ".2f"),format(np.std(spec_12), ".2f")])
            

insight_12_df = pd.DataFrame(insight_12)
insight_12_df.columns = ["k","AUC_ROC_mean", "AUC_ROC_std", "bal_acc_mean", "bal_acc_std","f1_mean","f1_std","sensitivity_mean","sensitivity_std","specificity_mean","specificity_std"]


### Task 3-4

In [ ]:
#3-4
insight_34 = []
CM_mean_34  = []
CM_sum_34  = []

CM_34  = []
cm_sum_34  = [[0 for j in range(2)] for i in range(2)]
AUC_ROC_34 = []
bal_acc_34 = []
f1_34 = []
sens_34=[]
spec_34=[]

for train_34_idx, validation_34_idx in skf.split(train_pat_34_df,y_train_pat_34):

    #training set crossvalidation
    x_34_train_fold,y_train_fold = training_set_creation(train_pat_34_df,train_34_idx,patients_new,X,y)

    #modifico label
    y_34_train_fold = y_train_fold.copy()
    y_34_train_fold.loc[y_train_fold == 3] = 0
    y_34_train_fold.loc[y_train_fold == 4] = 1

    #validation set crossvalidation
    x_validation_pat_34,y_validation_pat_34,imgXpat_34 = validation_set_creation (train_pat_34_df,y_train_pat_34,validation_34_idx,patients_new,X,y)        

    #robust scaler
    x_train_fold_scaled, x_validation_fold_scaled = scaling_robust (x_34_train_fold, x_validation_pat_34)                    

    #feature selection
    x_train_fold_sel,x_validation_fold_sel = feat_sel(x_train_fold_scaled, x_validation_fold_scaled, y_34_train_fold, k_34) 

    #smote
    x_train_fold_resample, y_train_fold_resample = resampling_smote (x_train_fold_sel, y_34_train_fold)

    # fit classifier
    classifier.fit(x_train_fold_resample, y_train_fold_resample)

    # prediction
    y_pred = classifier.predict(x_validation_fold_sel)
    y_prob = classifier.predict_proba(x_validation_fold_sel)

    #mjv patient
    y_pred_pat_mv_34, y_prob_pat_class1_34 = major_voting_binary_validation(y_pred, y_prob, imgXpat_34)
    # y_pred_pat_mv_34, y_prob_pat_mean_sum_34,y_prob_pat_class0_34,y_prob_pat_class1_34,*_= major_voting(y_pred, y_prob, imgXpat_34, 2)
    
    #metrics
    cm_sum_34 = np.array(cm_sum_34) + np.array(confusion_matrix(y_validation_pat_34, y_pred_pat_mv_34))
    CM_34.append(confusion_matrix(y_validation_pat_34, y_pred_pat_mv_34))
    AUC_ROC_34.append(roc_auc_score(y_validation_pat_34, np.array(y_prob_pat_class1_34)[:, 1]))
    bal_acc_34.append(balanced_accuracy_score(y_validation_pat_34, y_pred_pat_mv_34))
    f1_34.append((f1_score(y_validation_pat_34, y_pred_pat_mv_34, pos_label= 1)))
    sens_34.append(recall_score(y_validation_pat_34,y_pred_pat_mv_34))
    spec_34.append(calculate_specificity(y_validation_pat_34, y_pred_pat_mv_34, n_classes = 2)[0])

    
CM_mean_34 = np.stack(CM_34).mean(axis=0)
CM_sum_34.append(cm_sum_34)
insight_34.append([k_34,format(np.mean(AUC_ROC_34), ".2f"),format(np.std(AUC_ROC_34), ".2f"), 
            format(np.mean(bal_acc_34), ".2f"),format(np.std(bal_acc_34), ".2f"),
            format(np.mean(f1_34), ".2f"),format(np.std(f1_34), ".2f"),
            format(np.mean(sens_34), ".2f"),format(np.std(sens_34), ".2f"),
            format(np.mean(spec_34), ".2f"),format(np.std(spec_34), ".2f")])
            
insight_34_df = pd.DataFrame(insight_34)
insight_34_df.columns = ["k","AUC_ROC_mean", "AUC_ROC_std", "bal_acc_mean", "bal_acc_std","f1_mean","f1_std","sensitivity_mean","sensitivity_std","specificity_mean","specificity_std"]


## model fitting on whole training set + cascade of models on test set

### sets preparation

In [ ]:
################ 0-1234-5
#training
i = 0  
for pt in range(len(train_pat)): 
    indices_tr = [i for i, s in enumerate(patients_new) if train_pat[pt] in s]    #trainig indices in X and y corresponding to selected patient  
    x_rows_pat_tr = X.iloc[indices_tr]# X rows in training
    y_rows_pat_tr = y[indices_tr]# y rows in training
    if i == 0:# first patient -> initialize
        x_train = x_rows_pat_tr
        y_train = y_rows_pat_tr
        i = 1
    else: # from second patient on -> concatenate
        x_train = pd.concat([x_train,x_rows_pat_tr]) 
        y_train = pd.concat([y_train,y_rows_pat_tr]) 
#modify label
y_train_0_1234_5 = y_train.copy()
y_train_0_1234_5[(y_train==2)|(y_train==3)|(y_train==4)] = 1
y_train_0_1234_5[(y_train==5)] = 2

#test
i = 0  
n=0
imgXpat_test_0_1234_5 = []
for pt in range(len(test_pat)):
    indices_te = [i for i, s in enumerate(patients_new) if test_pat[pt] in s]# test indices in X and y corresponding to selected patient 
    x_rows_pat_te = X.iloc[indices_te]# X rows in test
    y_rows_pat_te = y[indices_te]# y rows in test
    if i == 0:# first patient -> initialize
        x_test = x_rows_pat_te
        y_test = y_rows_pat_te
        imgXpat_test_0_1234_5.append(len(indices_te))
        i = 1
    else:    # from second patient on -> concatenate
        x_test = pd.concat([x_test,x_rows_pat_te]) 
        y_test = pd.concat([y_test,y_rows_pat_te]) 
        imgXpat_test_0_1234_5.append(len(indices_te))
#modify label
y_test_0_1234_5 = y_test.copy()
y_test_0_1234_5[(y_test==2)|(y_test==3)|(y_test==4)] = 1
y_test_0_1234_5[(y_test==5)] = 2
y_test_pat_0_1234_5_global=y_test_pat.copy()
y_test_pat_0_1234_5 = y_test_pat.copy()
y_test_pat_0_1234_5[(y_test_pat==2)|(y_test_pat==3)|(y_test_pat==4)] = 1
y_test_pat_0_1234_5[(y_test_pat==5)] = 2


In [ ]:

################### 12-34
#training
i = 0
for pt in train_pat_12_34:
    indices = [i for i, s in enumerate(patients_new) if pt in s]#trainig indices in X and y corresponding to selected patient  
    x_rows_pat_tr = X.iloc[indices]# X rows in training
    y_rows_pat_tr = y.iloc[indices]# y rows in training
    if i == 0:# first patient -> initialize
        x_train_12_34 = x_rows_pat_tr
        y_train_12_34 = y_rows_pat_tr
        i = 1
    else:    # from second patient on -> concatenate
        x_train_12_34 = pd.concat([x_train_12_34,x_rows_pat_tr]) 
        y_train_12_34 = pd.concat([y_train_12_34,y_rows_pat_tr]) 
#modify label
y_train_12_34[(y_train_12_34==1)|(y_train_12_34==2)] = 0
y_train_12_34[(y_train_12_34==3)|(y_train_12_34==4)] = 1

################### 1-2
#training
i=0
for pt in train_pat_12:
    indices = [i for i, s in enumerate(patients_new) if pt in s]#trainig indices in X and y corresponding to selected patient  
    x_rows_pat_tr = X.iloc[indices]# X rows in training
    y_rows_pat_tr = y.iloc[indices]# y rows in training
    if i == 0:# first patient -> initialize
        x_train_12 = x_rows_pat_tr
        y_train_12 = y_rows_pat_tr
        i = 1
    else:    # from second patient on -> concatenate
        x_train_12 = pd.concat([x_train_12,x_rows_pat_tr]) 
        y_train_12 = pd.concat([y_train_12,y_rows_pat_tr]) 
#modify label
y_train_12[(y_train_12==1)] = 0
y_train_12[(y_train_12==2)] = 1

################### 3-4
#training
i = 0
for pt in train_pat_34:
    indices = [i for i, s in enumerate(patients_new) if pt in s]#trainig indices in X and y corresponding to selected patient  
    x_rows_pat_tr = X.iloc[indices]# X rows in training
    y_rows_pat_tr = y.iloc[indices]# y rows in training
    if i == 0:# first patient -> initialize
        x_train_34 = x_rows_pat_tr
        y_train_34 = y_rows_pat_tr
        i = 1
    else:    # from second patient on -> concatenate
        x_train_34 = pd.concat([x_train_34,x_rows_pat_tr]) 
        y_train_34 = pd.concat([y_train_34,y_rows_pat_tr]) 
#modify label
y_train_34[(y_train_34==3)] = 0
y_train_34[(y_train_34==4)] = 1


### n images in training and test

In [ ]:
# n img training
for i in range(6):
    print(i, np.sum(y_train==i))
print(len(y_train))

In [ ]:
# n img test
for i in range(6):
    print(i, np.sum(y_test==i))
print(len(y_test))

### re-training and test

### Task 0 - 1234 - 5

In [ ]:
insight_test_0_1234_5 = []
insight_test_12_34 = []
insight_test_12 = []
insight_test_34 = []

shap_seed = 27

############ 0-1234-5
#robust scaler
x_train_scaled, x_test_scaled = scaling_robust (x_train, x_test)    
#feature selection    
x_train_sel,x_test_sel = feat_sel_multi(x_train_scaled, x_test_scaled, y_train_0_1234_5, k_0_1234_5) 
print(x_train_sel.columns)
#smote
x_train_resample_0_1234_5, y_train_resample = resampling_smote (x_train_sel,y_train_0_1234_5)
#model fitting
classifier.fit(x_train_resample_0_1234_5, y_train_resample)
# #shap
background_data = shap.sample(x_train_resample_0_1234_5, 100)
explainer_train_0_1234_5 = shap.KernelExplainer(classifier.predict_proba, background_data) #explainer fitted on training set
shap_values_train_0_1234_5 = explainer_train_0_1234_5(x_train_resample_0_1234_5) # shap values calculated on training set 
shap_values_test_0_1234_5 = explainer_train_0_1234_5(x_test_sel) # shap value calculated on test set
#prediction
y_pred_test_0_1234_5 = classifier.predict(x_test_sel)
y_prob_test_0_1234_5 = classifier.predict_proba(x_test_sel)
#mjv patient
y_pred_pat_mv_test_0_1234_5, y_prob_pat_mv_test_0_1234_5 = major_voting_multiclass(y_pred_test_0_1234_5, y_prob_test_0_1234_5, imgXpat_test_0_1234_5)
# y_pred_pat_mv_test_0_1234_5, y_prob_pat_mean_sum_test_0_1234_5,y_prob_pat_class0_test_0_1234_5,y_prob_pat_class1_test_0_1234_5,y_prob_pat_class2_test_0_1234_5 = major_voting(y_pred_test_0_1234_5, y_prob_test_0_1234_5, imgXpat_test_0_1234_5, 3)


### Altri Task

In [ ]:

############# 12-34
#test set 12-34 task creation
# take just the patients correctly predicted in 0-1234-5 task and belonging to class 1-2-3-4 ->create 12-34 test set
filter_correct_prediction_12_34 = [np.transpose(y_pred_pat_mv_test_0_1234_5 == np.transpose(y_test_pat_0_1234_5))][0] 
pat_test_set_correct_12_34 = [i for indx,i in enumerate(test_pat) if filter_correct_prediction_12_34[indx] == True] 
pat_test_set_correct_12_34_df = pd.DataFrame(pat_test_set_correct_12_34)
y_pat_test_set_correct = [i for indx,i in enumerate(y_test_pat) if filter_correct_prediction_12_34[indx] == True]
y_pat_test_set_correct = np.concatenate(y_pat_test_set_correct, axis=0)
pat_test_set_12_34 = pat_test_set_correct_12_34_df.loc[(y_pat_test_set_correct == 1) | (y_pat_test_set_correct == 2) | (y_pat_test_set_correct == 3)| (y_pat_test_set_correct == 4)]
pat_test_set_12_34 = np.concatenate(pat_test_set_12_34.values, axis=0)
y_pat_test_12_34 = y_pat_test_set_correct[(y_pat_test_set_correct == 1) | (y_pat_test_set_correct == 2) | (y_pat_test_set_correct == 3)| (y_pat_test_set_correct == 4)]

#modify label
y_pat_test_12_34_binary = y_pat_test_12_34.copy()
y_pat_test_12_34_binary[(y_pat_test_12_34_binary == 1)|(y_pat_test_12_34_binary == 2)] = 0
y_pat_test_12_34_binary[(y_pat_test_12_34_binary == 3)|(y_pat_test_12_34_binary == 4)] = 1
#create set
i = 0
imgXpat_12_34_test = []
n=0
for pat_te in pat_test_set_12_34:
    indices_te_12_34 = [i for i, s in enumerate(patients_new) if pat_te in s]# test indices in X and y corresponding to selected patient
    n=n+len(indices_te_12_34)
    x_rows_pat_test = X.iloc[indices_te_12_34]# X rows in test
    y_rows_pat_test = y.iloc[indices_te_12_34]# y rows in test
    if i == 0:# first patient -> initialize
        x_test_set_12_34= x_rows_pat_test
        y_test_set_12_34 = y_rows_pat_test
        imgXpat_12_34_test.append(len(indices_te_12_34))
        i = 1
    else:    # from second patient on -> concatenate 
        x_test_set_12_34 = pd.concat([x_test_set_12_34,x_rows_pat_test]) 
        y_test_set_12_34 = pd.concat([y_test_set_12_34,y_rows_pat_test])         
        imgXpat_12_34_test.append(len(indices_te_12_34))
#modify label
y_test_set_12_34.loc[(y_test_set_12_34 == 1)|(y_test_set_12_34 == 2)] = 0
y_test_set_12_34.loc[(y_test_set_12_34 == 3)|(y_test_set_12_34 == 4)] = 1        
#robust scaler
x_train_scaled, x_test_scaled = scaling_robust (x_train_12_34, x_test_set_12_34)        
#feature selection
x_train_sel,x_test_sel_12_34 = feat_sel(x_train_scaled, x_test_scaled, y_train_12_34, k_12_34) 
print(x_train_sel.columns)
#smote
x_train_resample_12_34, y_train_resample = resampling_smote (x_train_sel, y_train_12_34)
# model fitting
classifier.fit(x_train_resample_12_34, y_train_resample)
# # shap 
background_data = shap.sample(x_train_resample_12_34, 100)
explainer_train_12_34 = shap.KernelExplainer(classifier.predict_proba, background_data)#explainer fitted on training set
shap_values_train_12_34 = explainer_train_12_34(x_train_resample_12_34)# shap values calculated on training set 
shap_values_test_12_34 = explainer_train_12_34(x_test_sel_12_34)# shap values calculated on test set 
#prediction
y_pred_test_12_34 = classifier.predict(x_test_sel_12_34)                    
y_prob_test_12_34 = classifier.predict_proba(x_test_sel_12_34)
#mjv patient
y_pred_pat_mv_test_12_34, y_prob_pat_mv_test_12_34 = major_voting_binary(y_pred_test_12_34, y_prob_test_12_34, imgXpat_12_34_test)
#y_pred_pat_mv_test_12_34, y_prob_pat_mean_sum_test_12_34,y_prob_pat_class0_test_12_34,y_prob_pat_class1_test_12_34,*_ = major_voting(y_pred_test_12_34, y_prob_test_12_34, imgXpat_12_34_test, 2)

############## 1-2
#test set 1-2 task creation
# take just the patients correctly predicted in 12-34 task and belonging to class 1-2 create 1-2 test set
filter_correct_prediction_single_class = [np.transpose(y_pred_pat_mv_test_12_34 == np.transpose(y_pat_test_12_34_binary))][0] 
pat_test_set_correct_single_class = [i for indx,i in enumerate(pat_test_set_12_34) if filter_correct_prediction_single_class[indx] == True] 
pat_test_set_correct_single_class_df = pd.DataFrame(pat_test_set_correct_single_class)
y_pat_test_set_correct_single_class = [i for indx,i in enumerate(y_pat_test_12_34) if filter_correct_prediction_single_class[indx] == True]
y_pat_test_set_correct_single_class = np.array(y_pat_test_set_correct_single_class)
pat_test_set_12 = pat_test_set_correct_single_class_df.loc[(y_pat_test_set_correct_single_class == 1) | (y_pat_test_set_correct_single_class == 2)]
if len(pat_test_set_12) != 0:
    # modify label
    y_pat_test_12 = y_pat_test_set_correct_single_class[(y_pat_test_set_correct_single_class == 1) | (y_pat_test_set_correct_single_class == 2)]
    y_pat_test_12[(y_pat_test_12 == 1)] = 0
    y_pat_test_12[(y_pat_test_12 == 2)] = 1
    #create set
    i = 0
    n=0
    imgXpat_12_test = []
    for pt_te in range(len(pat_test_set_12)):
        pat_te = ((str(pat_test_set_12.iloc[pt_te])).split())[1]
        indices_te_12 = [i for i, s in enumerate(patients_new) if pat_te in s]# test indices in X and y corresponding to selected patient
        n=n+len(indices_te_12)
        x_rows_pat_test = X.iloc[indices_te_12]# X rows in test
        y_rows_pat_test = y.iloc[indices_te_12]  # y rows in test
        if i == 0:# first patient -> initialize
            x_test_set_12= x_rows_pat_test
            y_test_set_12 = y_rows_pat_test
            imgXpat_12_test.append(len(indices_te_12))
            indices_test_12 = indices_te_12
            i = 1
        else:    # from second patient on -> concatenate
            x_test_set_12 = pd.concat([x_test_set_12,x_rows_pat_test]) 
            y_test_set_12 = pd.concat([y_test_set_12,y_rows_pat_test])        
            imgXpat_12_test.append(len(indices_te_12))
            indices_test_12 = np.concatenate([indices_test_12,indices_te_12])
    #modify label
    y_test_set_12.loc[(y_test_set_12 == 1)] = 0
    y_test_set_12.loc[(y_test_set_12 == 2)] = 1        
    #robust scaler
    x_train_scaled12, x_test_scaled12 = scaling_robust (x_train_12, x_test_set_12)        
    #feature selection
    x_train_sel12,x_test_sel_12 = feat_sel(x_train_scaled12, x_test_scaled12, y_train_12, k_12) 
    print(x_train_sel12.columns)
    #smote
    x_train_resample_12, y_train_resample12 = resampling_smote (x_train_sel12, y_train_12)
    #model fitting
    classifier.fit(x_train_resample_12, y_train_resample12)
    # #shap 
    background_data = shap.sample(x_train_resample_12, 100)
    explainer_train_12 = shap.KernelExplainer(classifier.predict_proba, background_data)#explainer fitted on training set
    shap_values_train_12 = explainer_train_12(x_train_resample_12)# shap values calculated on training set
    shap_values_test_12 = explainer_train_12(x_test_sel_12)# shap values calculated on test set
    #prediction
    y_pred_test_12 = classifier.predict(x_test_sel_12)                    
    y_prob_test_12 = classifier.predict_proba(x_test_sel_12)
    #mjv patient
    y_pred_pat_mv_test_12, y_prob_pat_mv_test_12 = major_voting_binary(y_pred_test_12, y_prob_test_12, imgXpat_12_test)
    #y_pred_pat_mv_test_12, y_prob_pat_mean_sum_test_12,y_prob_pat_class0_test_12,y_prob_pat_class1_test_12,*_ = major_voting(y_pred_test_12, y_prob_test_12, imgXpat_12_test, 2)

##### 3-4
#test set 3-4 task creation
# take just the patients correctly predicted in 12-34 task and belonging to class 3-4 -> create 3-4 test set
pat_test_set_34 = pat_test_set_correct_single_class_df.loc[(y_pat_test_set_correct_single_class == 3) | (y_pat_test_set_correct_single_class == 4)]
#modify label
y_pat_test_34 = y_pat_test_set_correct_single_class[(y_pat_test_set_correct_single_class == 3) | (y_pat_test_set_correct_single_class == 4)]
y_pat_test_34[(y_pat_test_34 == 3)] = 0
y_pat_test_34[(y_pat_test_34 == 4)] = 1
#create set
i = 0
n=0
imgXpat_34_test = []
for pt_te in range(len(pat_test_set_34)):
    pat_te = ((str(pat_test_set_34.iloc[pt_te])).split())[1]
    indices_te_34 = [i for i, s in enumerate(patients_new) if pat_te in s]# test indices in X and y corresponding to selected patient
    n=n+len(indices_te_34)
    x_rows_pat_test = X.iloc[indices_te_34]# X rows in test
    y_rows_pat_test = y.iloc[indices_te_34]  # y rows in test
    if i == 0: # first patient -> initialize
        x_test_set_34= x_rows_pat_test
        y_test_set_34 = y_rows_pat_test
        imgXpat_34_test.append(len(indices_te_34))
        indices_test_34 = indices_te_34
        i = 1
    else:    # from second patient on -> concatenate
        x_test_set_34 = pd.concat([x_test_set_34,x_rows_pat_test]) 
        y_test_set_34 = pd.concat([y_test_set_34,y_rows_pat_test])       
        imgXpat_34_test.append(len(indices_te_34))
        indices_test_34 = np.concatenate([indices_test_34,indices_te_34])
#modify label
y_test_set_34.loc[(y_test_set_34 == 3)] = 0
y_test_set_34.loc[(y_test_set_34 == 4)] = 1        
#robust scaler
x_train_scaled34, x_test_scaled34 = scaling_robust (x_train_34, x_test_set_34)        
#feature selection
x_train_sel34,x_test_sel_34 = feat_sel(x_train_scaled34, x_test_scaled34, y_train_34, k_34)
print(x_train_sel34.columns)
#smote
x_train_resample_34, y_train_resample34 = resampling_smote (x_train_sel34, y_train_34)
#model fitting
classifier.fit(x_train_resample_34, y_train_resample34)
# #shap
background_data = shap.sample(x_train_resample_34, 100)
explainer_train_34 = shap.KernelExplainer(classifier.predict_proba, background_data)#explainer fitted on training set
shap_values_train_34 = explainer_train_34(x_train_resample_34)# shap values calculated on training set
shap_values_test_34 = explainer_train_34(x_test_sel_34)# shap values calculated on test set
#prediction
y_pred_test_34 = classifier.predict(x_test_sel_34)                    
y_prob_test_34 = classifier.predict_proba(x_test_sel_34)
#mjv patient
y_pred_pat_mv_test_34, y_prob_pat_mv_test_34 = major_voting_binary(y_pred_test_34, y_prob_test_34, imgXpat_34_test)
#y_pred_pat_mv_test_34, y_prob_pat_mean_sum_test_34,y_prob_pat_class0_test_34,y_prob_pat_class1_test_34,*_ = major_voting(y_pred_test_34, y_prob_test_34, imgXpat_34_test, 2)
